# Interactive Volatility and GARCH Dashboard

Module: Financial Time Series

## Lesson summary

This dashboard connects volatility clustering, EWMA volatility, and GARCH recursion. Students adjust GARCH parameters and observe how persistence changes the speed at which volatility decays after shocks {cite}`engle1982autoregressive,bollerslev1986generalized`.

## Learning objectives

By the end of this dashboard, students should be able to:

- explain why volatility is time varying;
- simulate a GARCH(1,1)-style conditional volatility process;
- interpret $\alpha$, $\beta$, and persistence;
- compare EWMA volatility with conditional volatility;
- discuss why high persistence matters for risk forecasting.

## Volatility recursion

The dashboard compares two simple volatility updates. GARCH persistence comes from:

$$
\sigma_t^2=\omega+\alpha r_{t-1}^2+\beta\sigma_{t-1}^2,
$$

while EWMA volatility uses an exponential decay rule:

$$
\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2.
$$

Higher $\alpha+\beta$ or higher $\lambda$ means shocks decay more slowly.

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from ipywidgets import FloatSlider, IntSlider, interact
from plotly.subplots import make_subplots

from src.market_risk import ewma_volatility
from src.time_series_diagnostics import garch_persistence

RUN_INTERACTIVE_WIDGETS = os.getenv("RUN_INTERACTIVE_WIDGETS", "1") == "1"

## Simulation helper

In [ ]:
def simulate_garch_returns(periods=750, omega=0.000002, alpha=0.08, beta=0.90, seed=91):
    if alpha + beta >= 0.999:
        beta = 0.999 - alpha
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2023-01-02", periods=periods)
    variance = np.empty(periods)
    returns = np.empty(periods)
    variance[0] = omega / max(1 - alpha - beta, 1e-6)
    returns[0] = rng.normal(0, np.sqrt(variance[0]))

    for idx in range(1, periods):
        variance[idx] = omega + alpha * returns[idx - 1] ** 2 + beta * variance[idx - 1]
        returns[idx] = rng.normal(0, np.sqrt(variance[idx]))

    return pd.DataFrame(
        {
            "return": returns,
            "conditional_volatility": np.sqrt(variance),
        },
        index=dates,
    )

## Interactive dashboard

Run this cell in JupyterLab with `uv run jupyter lab`.

In [ ]:
def plot_volatility_dashboard(
    omega=0.000002,
    alpha=0.08,
    beta=0.90,
    lambda_=0.94,
    periods=750,
):
    simulation = simulate_garch_returns(
        periods=periods,
        omega=omega,
        alpha=alpha,
        beta=beta,
        seed=91,
    )
    simulation["ewma_volatility"] = ewma_volatility(simulation["return"], lambda_=lambda_)

    persistence = garch_persistence(alpha, min(beta, 0.999 - alpha))
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=("Returns", "Conditional volatility versus EWMA volatility"),
    )
    fig.add_trace(
        go.Scatter(x=simulation.index, y=simulation["return"], mode="lines", name="return"),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=simulation.index,
            y=simulation["conditional_volatility"],
            mode="lines",
            name="GARCH volatility",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=simulation.index,
            y=simulation["ewma_volatility"],
            mode="lines",
            name="EWMA volatility",
        ),
        row=2,
        col=1,
    )
    fig.update_layout(
        title=f"Volatility dashboard | alpha + beta = {persistence:.3f}",
        template="plotly_white",
        height=650,
    )
    fig.update_yaxes(tickformat=".1%", row=1, col=1)
    fig.update_yaxes(tickformat=".1%", row=2, col=1)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


if RUN_INTERACTIVE_WIDGETS:
    interact(
        plot_volatility_dashboard,
        omega=FloatSlider(value=0.000002, min=0.0000005, max=0.000010, step=0.0000005, readout_format=".7f"),
        alpha=FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01, readout_format=".2f"),
        beta=FloatSlider(value=0.90, min=0.50, max=0.98, step=0.01, readout_format=".2f"),
        lambda_=FloatSlider(value=0.94, min=0.80, max=0.99, step=0.01, readout_format=".2f"),
        periods=IntSlider(value=750, min=252, max=1250, step=126),
    );
else:
    plot_volatility_dashboard()

## Model limitations

- The dashboard isolates GARCH-style persistence, so it does not represent jumps, leverage effects, or changing parameters.
- Simulated returns are useful for intuition but should not be treated as calibrated market forecasts.
- Persistence near one can make volatility shocks appear long-lived even when the real regime later changes.